In [ ]:
import pandas as pd
# pd.read_csv('stats-aws-latest.csv').to_pickle('stats-aws-latest.pkl')
# pd.read_csv('stats-gcp-latest.csv').to_pickle('stats-gcp-latest.pkl')

In [3]:
# create a chart that will render values in the dataframe
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

df = pd.read_pickle('../stats-all-latest.pkl')
print(f"Processing {df.size} datapoints...")

# Convert the 'isoDate' column to datetime objects
df['isoDate'] = pd.to_datetime(df['!isoDate'])
df = df.set_index("isoDate").sort_index()


# Extract the hour from the 'isoDate'
df["dow"] = df.index.day_of_week
df["hod"] = df.index.hour
df["local_hod"] = (df.index.hour - 5) % 24
df["how"] = (df["dow"] * 24) + df["hod"]

# Filter data for 'heif' output format
df_heif = df[df['outputFormat'] == 'heif']

sns.set_theme(style="whitegrid")

# Group by the specified columns and calculate the trimmed mean of 'durationImageProcessingMs'
trimmed_mean = lambda x: stats.trim_mean(x, 0.1)

# Group by the specified columns and calculate the mean 'durationImageProcessingMs'
grouped_df = df_heif.groupby(['imageName', 'outputWidth', 'providerMemoryAssociated', 'local_hod'])['durationProcessingMs'].agg(trimmed_mean).reset_index()
for name, group in grouped_df:
    print(f"Processing {name}, {group.size} datapoints")

# Plotting the results
for name, group in grouped_df.groupby(['imageName', 'outputWidth', 'providerMemoryAssociated']):
    print(f"Processing {name}, {group.size} datapoints")
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(group['local_hod'], group['durationImageProcessingMs'], label=str(name))

    ax.set_xlabel("Hour")
    ax.set_ylabel("Average durationImageProcessingMs")
    ax.set_title(f"Average durationImageProcessingMs by Hour for {name}")
    # ax.legend(title="Group", loc='upper right')

    # Format y-axis labels to show values in percent, relative to min value
    # ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=group['durationImageProcessingMs'].mean()))
    
    # Set y axis to be between 95 and 105% of the mean
    # ax.set_ylim(group['durationImageProcessingMs'].mean() * 0.98, group['durationImageProcessingMs'].mean() * 1.02)

    plt.tight_layout()
    plt.show()

    # snsplot = sns.displot(data=group, x=group['hod'], y=group['durationImageProcessingMs'], binwidth=1)
    # fig2 = snsplot.ax.get_figure()
    # fig2.set_size_inches(10, 6)
    # # fig2.savefig(f"chart-{name}.png")
    # fig2.show()





Processing 3968765 datapoints...


ValueError: too many values to unpack (expected 2)